# Downloading GMTED2010 Elevation

Source: [USGS/GMTED2010_FULL](https://developers.google.com/earth-engine/datasets/catalog/USGS_GMTED2010_FULL). Global coverage, extends to **84°N** (covers polar regions beyond SRTM's 60°N limit). Band `mea` (mean elevation) in meters.

Export scale = **100m** to match other elevation/environmental downloads. GMTED2010 blends SRTM with ASTER GDEM and other sources to fill gaps, so resolution trade-off is acceptable for zonal statistics over language polygons.

Exports go to Google Drive via Earth Engine. Monitor at the [GEE Task Manager](https://code.earthengine.google.com/tasks).

In [2]:
import ee
import geemap

ee.Authenticate()  # run once if not already authenticated
ee.Initialize()

In [3]:
# GMTED2010 covers 84°N to 56°S (includes polar regions)
gmted_bbox = ee.Geometry.BBox(-180, -56, 180, 84)

# Band 'mea' is mean elevation (available bands: be75, std, min, med, mea, max, dsc)
elevation = ee.Image("USGS/GMTED2010_FULL").select("mea")

print(elevation.getInfo()["id"])  # sanity-check

USGS/GMTED2010_FULL


In [4]:
# Quick preview
vis_params = {
    "min": 0,
    "max": 4000,
    "palette": ["000080", "0000ff", "00ffff", "00ff00", "ffff00", "ff8000", "ff0000", "ffffff"],
}

Map = geemap.Map(center=[0, 0], zoom=2)
Map.addLayer(elevation.clip(gmted_bbox), vis_params, "GMTED2010 Mean Elevation (m)")
Map.addLayer(gmted_bbox, {}, "Region")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [5]:
# Export to Google Drive at 100m resolution
task = ee.batch.Export.image.toDrive(
    image=elevation,
    description="gmted2010_elevation_100m",
    folder="GEE_exports",
    fileNamePrefix="gmted2010_elevation_100m",
    region=gmted_bbox,
    scale=100,
    crs="EPSG:4326",
    maxPixels=1e13,
)

task.start()
print("Export task started:", task.id)

Export task started: P3A7NZVT5DD4NI7L2L6TPWSK


### NOTE: track the export at the [GEE Task Manager](https://code.earthengine.google.com/tasks)

Suggest placing output tiles in `maps/raw/GMTED2010/` in the Dropbox project folder, alongside the other raster inputs.